In [1]:
import ee
ee.Initialize(project='mapbiomas-bolivia')

# ----------------------------
# Inputs
# ----------------------------

# Vector sources
vetor_substancias_col9 = ee.FeatureCollection(
    'users/julcansado/Solved/MapBiomas/substances/mined_substances_col9'
)

vetor_substancias_col10 = ee.FeatureCollection(
    'projects/solved-mb10/assets/LANDSAT/MINING/col10_substances_vector'
)

# Mining classification ImageCollection (COLLECTION-10, version 6)
ic = ee.ImageCollection(
    "projects/mapbiomas-brazil/assets/LAND-COVER/COLLECTION-10/MINING/classification-ft"
).filter(ee.Filter.eq('version', '6'))

# Pick the 2024-6 image as reference for projection/grid
ref2024 = ic.filter(ee.Filter.eq('year', 2024)) \
            .filter(ee.Filter.eq('version', '6')) \
            .first()

# ----------------------------
# Prepare buffered vectors
# ----------------------------

def buffer_col9(f):
    return f.set('JOIN_class', ee.Number.parse(f.get('JOIN_class'))) \
            .buffer(200)  # 200 m buffer

col9_buffered = vetor_substancias_col9.map(buffer_col9)

def buffer_col10(f):
    return f.set('classifica', ee.Number.parse(f.get('classifica'))) \
            .buffer(60)  # 60 m buffer

col10_buffered = vetor_substancias_col10.map(buffer_col10)

# ----------------------------
# Rasterize vectors
# ----------------------------

col9_img = ee.Image(0).byte().paint(
    featureCollection=col9_buffered,
    color='JOIN_class'
)

col10_img = ee.Image(0).byte().paint(
    featureCollection=col10_buffered,
    color='classifica'
)

# Combine masks: col9 first, fill with col10 where col9 is 0
combined_mask = col9_img.where(col9_img.eq(0), col10_img)

# ----------------------------
# Process each year image
# ----------------------------

def process_image(img):
    year = ee.Number(img.get('year'))
    year_mask = img.gt(0)
    masked = year_mask.multiply(combined_mask)

    bandName = ee.String('classification_').cat(year.format('%.0f'))
    masked = masked.rename(bandName)

    return masked

processed_ic = ic.map(process_image)

# 1. Create the multiband image using toBands()
multiband_with_prefix = processed_ic.toBands()

# 2. Get the list of current, prefixed band names
oldNames = multiband_with_prefix.bandNames()

# 3. Remove the prefix from each name
newNames = oldNames.map(lambda name: ee.String(name).split('_').slice(1).join('_'))

# 4. Rename bands
multiband_renamed = multiband_with_prefix.select(oldNames, newNames)

# Set properties
multiband_renamed = multiband_renamed \
    .set('theme', 'MINING') \
    .set('collection_id', 10.0) \
    .set('source', 'Solved') \
    .set('territory', 'BRAZIL') \
    .set('version', '7')

print('Image with renamed bands:', multiband_renamed.getInfo())

# ----------------------------
# Export
# ----------------------------

task = ee.batch.Export.image.toAsset(
    image=multiband_renamed,
    description='classification_with_mask_multiband_V7',
    assetId='projects/solved-mb10/assets/LANDSAT/MINING/col10_substances/classification_multiband_V7',
    region=col10_buffered.geometry().bounds(),
    scale=30,
    maxPixels=1e13
)
task.start()

print('Export started')


/Users/joaosiqueira/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Image with renamed bands: {'type': 'Image', 'bands': [{'id': 'classification_1985', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'dimensions': [142846, 141238], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647, 0, -71.60336382424089, 0, -0.00026949458523585647, 5.208521848853398]}, {'id': 'classification_1986', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'dimensions': [142846, 141238], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647, 0, -71.60336382424089, 0, -0.00026949458523585647, 5.208521848853398]}, {'id': 'classification_1987', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'dimensions': [142846, 141238], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647, 0, -71.60336382424089, 0, -0.00026949458523585647, 5.208521848853398]}, {'id': 'classification_1988', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'dimensions': [142846, 141238], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647